# RLSP — Qwen2.5-1.5B — Math — Dynamic Alpha


In [ ]:
!nvidia-smi

## 1. Setup & Clone

In [ ]:
print("Hello World")

In [ ]:
import os
print("Staring")
os.chdir('/teamspace/studios/this_studio')
!rm -rf /teamspace/studios/this_studio/Emergence-of-Thinking
!git clone https://github.com/GuanghaoYe/Emergence-of-Thinking.git
os.chdir('/teamspace/studios/this_studio/Emergence-of-Thinking')

print('---> Fixing Ray version...')
for root, dirs, files in os.walk('.'):
    if '.git' in root:
        continue
    for file in files:
        if file.endswith(('.txt', '.py', '.toml')):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, 'r') as f:
                    content = f.read()
                if 'ray' in content.lower() and '2.12.0' in content:
                    with open(filepath, 'w') as f:
                        f.write(content.replace('2.12.0', '2.31.0'))
                    print(f'  Fixed: {filepath}')
            except:
                pass

print('\n---> 0/2: Installing PyTorch...')
!pip install torch torchvision torchaudio

print('\n---> 1/2: vLLM, Ray, Deepspeed, PEFT, Flask...')
!pip install vllm==0.6.3 deepspeed peft wandb bitsandbytes "ray[default]" flask

print('\n---> 2/2: flash-attn...')
!pip install flash-attn --no-cache-dir --no-build-isolation || echo 'flash-attn failed, continuing'

In [ ]:
import os
print("Staring")
os.chdir('/teamspace/studios/this_studio')
!rm -rf /teamspace/studios/this_studio/Emergence-of-Thinking
!git clone https://github.com/GuanghaoYe/Emergence-of-Thinking.git
os.chdir('/teamspace/studios/this_studio/Emergence-of-Thinking')

print('---> Fixing Ray version...')
for root, dirs, files in os.walk('.'):
    if '.git' in root:
        continue
    for file in files:
        if file.endswith(('.txt', '.py', '.toml')):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, 'r') as f:
                    content = f.read()
                if 'ray' in content.lower() and '2.12.0' in content:
                    with open(filepath, 'w') as f:
                        f.write(content.replace('2.12.0', '2.31.0'))
                    print(f'  Fixed: {filepath}')
            except:
                pass

print('\n---> 0/3: Installing PyTorch...')
!pip install torch torchvision torchaudio

print('\n---> 1/3: Core project...')
!pip install -e . --no-build-isolation

print('\n---> 2/3: vLLM, Ray, Deepspeed, PEFT, Flask...')
!pip install vllm==0.6.3 deepspeed peft wandb bitsandbytes "ray[default]" flask

print('\n---> 3/3: flash-attn...')
!pip install flash-attn --no-cache-dir --no-build-isolation || echo 'flash-attn failed, continuing'

print('Setup complete!')

## 2. Transformers fix

In [ ]:
!pip install transformers==4.45.2 accelerate -q

## 3. Download Qwen2.5-1.5B-Instruct

In the following code cell you should attach your Hugging Face token

In [ ]:
import os
# Attach your Hugging Face token
os.environ["HF_TOKEN"] = "....."

model_path = '/teamspace/studios/this_studio/models/Qwen2.5-1.5B-Instruct'
if not os.path.exists(model_path):
    !pip install huggingface_hub --quiet
    !huggingface-cli download Qwen/Qwen2.5-1.5B-Instruct \
        --local-dir /teamspace/studios/this_studio/models/Qwen2.5-1.5B-Instruct
    print('Model downloaded')
else:
    print('Model already present, skipping')

In [ ]:
!pip install latex2sympy2

## 4. Dataset & Directories

In [ ]:
import os, json

os.makedirs('/tmp/code', exist_ok=True)
os.makedirs('/teamspace/studios/this_studio/Emergence-of-Thinking/logs/openrlhf_train_ppo', exist_ok=True)
os.makedirs('/teamspace/studios/this_studio/output_math_dynamic', exist_ok=True)

# Transforming into Qwen format
with open('/teamspace/studios/this_studio/Emergence-of-Thinking/data/math_formatted.jsonl') as f_in, \
     open('/teamspace/studios/this_studio/Emergence-of-Thinking/data/math_qwen.jsonl', 'w') as f_out:
    for line in f_in:
        item = json.loads(line)
        item['input'] = f"<|im_start|>user\nProblem: {item['problem']}<|im_end|>\n<|im_start|>assistant\n"
        f_out.write(json.dumps(item) + '\n')

print('math_qwen.jsonl created with Qwen chat tokens')

# Verify
with open('/teamspace/studios/this_studio/Emergence-of-Thinking/data/math_qwen.jsonl') as f:
    sample = json.loads(f.readline())
print('Sample input:', sample['input'][:150])

## 5. Clean old checkpoints

In [ ]:
import shutil

shutil.rmtree('/teamspace/studios/this_studio/output_math_dynamic', ignore_errors=True)
shutil.rmtree('/tmp/ray', ignore_errors=True)

import os
os.makedirs('/teamspace/studios/this_studio/output_math_dynamic', exist_ok=True)

total, used, free = shutil.disk_usage('/teamspace/studios/this_studio')
print(f'Persistent storage -- Free: {free/1e9:.1f} GB')
print('Ready for training')

## 6. Patches

In [ ]:
# Patch 1: DeepSpeed lm_head bug
filepath = '/teamspace/studios/this_studio/Emergence-of-Thinking/openrlhf/utils/deepspeed/deepspeed.py'
with open(filepath, 'r') as f:
    lines = f.readlines()

patched = False
new_lines = []
for line in lines:
    if 'assert state_dict_keys.issubset(' in line and not patched:
        indent = len(line) - len(line.lstrip())
        new_lines.append(' ' * indent + 'state_dict_keys -= {"base_model.model.lm_head.weight"}\n')
        patched = True
    new_lines.append(line)

if patched:
    with open(filepath, 'w') as f:
        f.writelines(new_lines)
    print('DeepSpeed patch applied')
else:
    print('Already patched or pattern not found')

# Patch 2: Ray GPU override
import ray._private.resource_and_label_spec as _spec
import inspect, pathlib

fpath = pathlib.Path(inspect.getfile(_spec))
src = fpath.read_text()

old = """            raise ValueError(
                f"Attempting to start raylet with {num_accelerators} "
                f"{accelerator_resource_name}, "
                f"{accelerator_manager.get_visible_accelerator_ids_env_var()} "
                f"contains {visible_accelerator_ids}."
            )"""

new = """            pass  # patched: allow virtual GPU override"""

if old in src:
    fpath.write_text(src.replace(old, new))
    print('Ray patch applied')
else:
    print('Ray patch: already applied or pattern mismatch')

## 7. Dynamic Alpha ORM Server

The main part of our expansion. Hyperparameter α starts from 0.2 (exploration) and reaches 0.9 (correctness).

In [ ]:
import os

server_code = r'''
import concurrent.futures
import datetime
import inspect
import io
import json
import logging
import multiprocessing
import os
import re
import signal
import sys
import time
from pathlib import Path
from typing import Dict, List

from datasets import Dataset, DatasetDict, load_dataset
from fastapi import FastAPI, Request
from pydantic import BaseModel
from tqdm import tqdm

from evaluation.grader import math_equal
from openrlhf.cli.gpt_reward import grade_cot

os.environ["TOKENIZERS_PARALLELISM"] = "false"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

args = None
ground_truth_dict = {}
tokenizer = None

# Dynamic Alpha
MAX_QUERIES    = 800
ALPHA_START    = 0.1
ALPHA_END      = 0.9

# we use a shared counter between the processes (multiprocessing.Value),
# if we used plain global int -- every child process would always see 0.
TOTAL_QUERIES = multiprocessing.Value("i", 0)

def get_dynamic_alpha(current, max_q):
    t = min(current / max(max_q, 1), 1.0)
    return ALPHA_START + t * (ALPHA_END - ALPHA_START)

class RequestLogger:
    def __init__(self, log_dir="logs"):
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(exist_ok=True)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_file = self.log_dir / f"requests_{timestamp}.jsonl"
        self.log_file.touch()
        logger.info(f"Logging requests to: {self.log_file}")

    def log_request(self, request_data, reward, processing_time):
        log_entry = {
            "timestamp": datetime.datetime.now().isoformat(),
            "request": request_data,
            "reward": reward,
            "processing_time": processing_time,
        }
        with open(self.log_file, "a") as f:
            f.write(json.dumps(log_entry) + "\n")


def parse_query(sequence):
    global args
    if 'llama' in args.model_name.lower():
        query = sequence.split("Problem: ")[1].split("<|eot_id|><|start_header_id|>assistant<|end_header_id|>")[0].strip()
        response = sequence.split("<|start_header_id|>assistant<|end_header_id|>\n\n")[1].strip()
    elif 'qwen' in args.model_name.lower():
        query = sequence.split("<|im_start|>user\nProblem:")[1].split("<|im_end|>")[0].strip()
        response = sequence.split("<|im_start|>assistant\n")[1].strip()
    elif 'cot' in args.model_name.lower():
        query = sequence.split("Question: ")[1].split("\nAnswer: ")[0].strip()
        response = sequence.split("Answer: ")[1].strip()
    else:
        raise ValueError("Model name not recognized")
    return query, response


def extract_answer_and_reasoning(text):
    pattern = r'\\boxed{([^{}]*(?:{[^{}]*}[^{}]*)*)}'
    match = re.search(pattern, text)
    if match:
        return match.group(1), text[:match.start()]
    else:
        return None, None


def _compute_single_reward(seq: str) -> float:
    global args, ground_truth_dict, tokenizer, TOTAL_QUERIES

    try:
        tic = time.time()
        query, response = parse_query(seq)
        ground_truth = ground_truth_dict[re.sub(r'[^a-zA-Z0-9]', '', query)]

        answer, reasoning = extract_answer_and_reasoning(response)

        if answer is None:
            outcome = -0.5
        else:
            outcome = float(math_equal(ground_truth, answer, timeout=True))

        # Dynamic Alpha reward (shared counter, thread/process-safe)
        with TOTAL_QUERIES.get_lock():
            TOTAL_QUERIES.value += 1
            current_q = TOTAL_QUERIES.value
        alpha = get_dynamic_alpha(current_q, MAX_QUERIES)

        if reasoning is None:
            reasoning_length = max(len(response) / 4, 1)
        else:
            reasoning_length = max(len(reasoning) / 4, 1)

        exploration_reward = -args.length_penalty / reasoning_length
        reward = alpha * outcome + (1.0 - alpha) * exploration_reward

        t_pct = min(current_q / MAX_QUERIES * 100, 100)
        if outcome == 1.0:
            print(f"CORRECT! [q={current_q:>4}/{MAX_QUERIES} {t_pct:5.1f}%] "
                  f"α={alpha:.3f} | answer={answer} | len~{reasoning_length:.0f} | r={reward:.4f}", flush=True)
        else:
            print(f"[q={current_q:>4}/{MAX_QUERIES} {t_pct:5.1f}%] "
                  f"α={alpha:.3f} | outcome={outcome:.1f} | len~{reasoning_length:.0f} | r={reward:.4f}", flush=True)

        duration = time.time() - tic
        print(time.ctime(), f"Duration: {duration:.2f}s", flush=True)

    except Exception as e:
        reward = 0.0
        print(f"Error: {e}", flush=True)

    return reward


def calculate_reward(data: dict):
    sequence = data.get("query", "")
    if isinstance(sequence, str):
        sequence = [sequence]
    with concurrent.futures.ProcessPoolExecutor() as executor:
        rewards = list(executor.map(_compute_single_reward, sequence))
    return rewards


app = FastAPI()
request_logger = None

@app.on_event("startup")
async def startup_event():
    global request_logger
    request_logger = RequestLogger(log_dir=args.log_dir)

@app.post("/get_reward")
async def get_reward(data: Request):
    start_time = time.time()
    try:
        data = await data.json()
        reward = calculate_reward(data)
        processing_time = time.time() - start_time
        request_logger.log_request(data, reward, processing_time)
    except Exception as e:
        print(f"Error: {str(e)}", flush=True)
        reward = 0.0
    return {"rewards": reward}


if __name__ == "__main__":
    import argparse
    import uvicorn

    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset",        type=str,   required=True)
    parser.add_argument("--model_name",     type=str,   required=True)
    parser.add_argument("--log_dir",        type=str,   default="logs")
    parser.add_argument("--length_penalty", type=float, default=20.0)
    parser.add_argument("--use_gpt",        type=float, default=False)

    args = parser.parse_args()
    os.makedirs(args.log_dir, exist_ok=True)

    dataset = load_dataset(args.dataset)

    if isinstance(dataset, DatasetDict):
        for split in dataset:
            for example in dataset[split]:
                query = re.sub(r"[^a-zA-Z0-9]", "", example["problem"])
                ground_truth_dict[query] = example["answer"]
    elif isinstance(dataset, Dataset):
        for example in dataset:
            query = re.sub(r"[^a-zA-Z0-9]", "", example["problem"])
            ground_truth_dict[query] = example["answer"]

    print(f"Loaded {len(ground_truth_dict)} problems", flush=True)
    print(f"Dynamic Alpha: {ALPHA_START} → {ALPHA_END} over {MAX_QUERIES} queries", flush=True)
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

path = '/teamspace/studios/this_studio/Emergence-of-Thinking/openrlhf/cli/orm_server_dynamic.py'
with open(path, 'w') as f:
    f.write(server_code)
print("orm_server_dynamic.py written")

## 8. PPO Training Script

In [ ]:
import os

script_content = r"""#!/bin/bash
set -x
ray stop --force 2>/dev/null || true
sleep 3
CUDA_VISIBLE_DEVICES=0 \
RAY_EXPERIMENTAL_NOSET_CUDA_VISIBLE_DEVICES=1 \
RAY_DISABLE_GPU_USAGE_STATS=1 \
ray start --head --num-gpus 2 \
    --dashboard-host=0.0.0.0 \
    --include-dashboard=True \
    --disable-usage-stats \
    --object-store-memory=2000000000
echo "Waiting 15 seconds..."
sleep 15s
ray job submit --address="http://127.0.0.1:8265" \
    --runtime-env-json='{"working_dir": "/teamspace/studios/this_studio/Emergence-of-Thinking"}' \
    -- python -m openrlhf.cli.train_ppo_ray \
    --ref_num_nodes 1 \
    --ref_num_gpus_per_node 1 \
    --reward_num_nodes 1 \
    --reward_num_gpus_per_node 1 \
    --critic_num_nodes 1 \
    --critic_num_gpus_per_node 1 \
    --actor_num_nodes 1 \
    --actor_num_gpus_per_node 1 \
    --vllm_num_engines 0 \
    --vllm_tensor_parallel_size 1 \
    --colocate_actor_ref \
    --colocate_critic_reward \
    --pretrain /teamspace/studios/this_studio/models/Qwen2.5-1.5B-Instruct \
    --reward_pretrain /teamspace/studios/this_studio/models/Qwen2.5-1.5B-Instruct \
    --save_path /teamspace/studios/this_studio/output_math_dynamic \
    --micro_train_batch_size 2 \
    --train_batch_size 8 \
    --micro_rollout_batch_size 4 \
    --rollout_batch_size 32 \
    --max_samples 480 \
    --max_epochs 1 \
    --num_episodes 20 \
    --prompt_max_len 512 \
    --generate_max_len 512 \
    --zero_stage 2 \
    --bf16 \
    --actor_learning_rate 2e-7 \
    --critic_learning_rate 2e-6 \
    --init_kl_coef 0.01 \
    --lambd 0.95 \
    --save_steps 9999 \
    --save_hf_ckpt \
    --disable_ds_ckpt \
    --prompt_data /teamspace/studios/this_studio/Emergence-of-Thinking/data/math_qwen.jsonl \
    --input_key input \
    --normalize_reward \
    --gradient_checkpointing \
    --remote_rm_url http://127.0.0.1:8000/get_reward \
    --temperature 0.7 \
    --max_ckpt_num 5 \
    --lora_rank 64 \
    --lora_alpha 128 \
    --target_modules q_proj k_proj v_proj o_proj
"""

script_path = '/teamspace/studios/this_studio/Emergence-of-Thinking/train_ppo_dynamic.sh'
with open(script_path, 'w') as f:
    f.write(script_content)
os.chmod(script_path, 0o755)
print('Script written')

## 9. Start Dynamic Alpha ORM Server

In [ ]:
import subprocess, time, sys, threading, json as _json, urllib.request

!pkill -f orm_server_dynamic || true
time.sleep(2)

log_path = '/teamspace/studios/this_studio/orm_server_dynamic.log'
log_file = open(log_path, 'w')

proc = subprocess.Popen(
    [sys.executable, '-m', 'openrlhf.cli.orm_server_dynamic',
     '--dataset',        'evaluation/data/math',
     '--model_name',     '/teamspace/studios/this_studio/models/Qwen2.5-1.5B-Instruct',
     '--log_dir',        './logs/openrlhf_train_ppo',
     '--length_penalty', '20',
     '--use_gpt',        '0'],
    cwd='/teamspace/studios/this_studio/Emergence-of-Thinking',
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print(f'ORM server PID: {proc.pid}')

# Background thread: reads log file live and prints in the notebook
def _tail_log(path, stop_event):
    with open(path, 'r') as f:
        while not stop_event.is_set():
            line = f.readline()
            if line:
                print(line.rstrip(), flush=True)
            else:
                time.sleep(0.5)

stop_tail = threading.Event()
tail_thread = threading.Thread(target=_tail_log, args=(log_path, stop_tail), daemon=True)
tail_thread.start()

time.sleep(10)
try:
    dummy = _json.dumps({'query': ['test \\boxed{42}']}).encode()
    req = urllib.request.Request('http://127.0.0.1:8000/get_reward', data=dummy,
        headers={'Content-Type': 'application/json'}, method='POST')
    resp = urllib.request.urlopen(req, timeout=10)
    print(f'Server UP: {resp.read().decode()[:100]}')
except Exception as e:
    print(f'Error: {e}')

In [ ]:
import pathlib, shutil, glob

print("=" * 50)
print("Applying all patches...")
print("=" * 50)

# PATCH 1: DeepSpeed lm_head
filepath = '/teamspace/studios/this_studio/Emergence-of-Thinking/openrlhf/utils/deepspeed/deepspeed.py'
with open(filepath, 'r') as f:
    lines = f.readlines()

patched = False
new_lines = []
for line in lines:
    if 'assert state_dict_keys.issubset(' in line and not patched:
        indent = len(line) - len(line.lstrip())
        new_lines.append(' ' * indent + 'state_dict_keys -= {"base_model.model.lm_head.weight"}\n')
        patched = True
    new_lines.append(line)

if patched:
    with open(filepath, 'w') as f:
        f.writelines(new_lines)
    print("Patch 1: DeepSpeed lm_head applied")
else:
    print("Patch 1: DeepSpeed already patched")

# PATCH 2: Ray resource_and_label_spec
fpath = pathlib.Path('/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ray/_private/resource_and_label_spec.py')
src = fpath.read_text()

old = '            raise ValueError(\n                f"Attempting to start raylet with {num_accelerators} "\n                f"{accelerator_resource_name}, "\n                f"but {accelerator_manager.get_visible_accelerator_ids_env_var()} "\n                f"contains {visible_accelerator_ids}."\n            )'
new = '            pass  # patched: allow virtual GPU override'

if old in src:
    fpath.write_text(src.replace(old, new))
    print("Patch 2: Ray GPU override applied")
else:
    print("Patch 2: Ray GPU override already patched")

# PATCH 3: Ray utils.py accelerator IDs
fpath3 = pathlib.Path('/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ray/_private/utils.py')
src3 = fpath3.read_text()

old3 = '    for resource_name, accelerator_ids in (\n        ray.get_runtime_context().get_accelerator_ids().items()\n    ):'
new3 = '''    try:
        _acc_items = ray.get_runtime_context().get_accelerator_ids().items()
    except Exception:
        _acc_items = {}.items()
    for resource_name, accelerator_ids in _acc_items:'''

if old3 in src3:
    fpath3.write_text(src3.replace(old3, new3))
    print("Patch 3: Ray utils.py accelerator IDs applied")
else:
    print("Patch 3: Ray utils.py already patched")

# PATCH 4: launcher.py LOCAL_RANK
filepath4 = '/teamspace/studios/this_studio/Emergence-of-Thinking/openrlhf/trainer/ray/launcher.py'
with open(filepath4, 'r') as f:
    src4 = f.read()

old4a = 'os.environ["LOCAL_RANK"] = str(ray.get_gpu_ids()[0]) if ray_noset_visible_devices() else "0"'
old4b = '_gpu_ids = ray.get_gpu_ids()\n        os.environ["LOCAL_RANK"] = str(_gpu_ids[0]) if (ray_noset_visible_devices() and _gpu_ids) else "0"'

new4 = '''try:
            _gpu_ids = ray.get_gpu_ids()
            os.environ["LOCAL_RANK"] = str(_gpu_ids[0]) if (ray_noset_visible_devices() and _gpu_ids) else "0"
        except Exception:
            os.environ["LOCAL_RANK"] = "0"'''

if old4a in src4:
    with open(filepath4, 'w') as f:
        f.write(src4.replace(old4a, new4))
    print("Patch 4: launcher.py LOCAL_RANK applied")
elif old4b in src4:
    with open(filepath4, 'w') as f:
        f.write(src4.replace(old4b, new4))
    print("Patch 4: launcher.py LOCAL_RANK (v2) applied")
else:
    print("Patch 4: launcher.py already patched")

# PATCH 5: Clear Ray cache
cleared = 0
for path in glob.glob('/tmp/ray/session_*/runtime_resources/working_dir_files/'):
    shutil.rmtree(path, ignore_errors=True)
    cleared += 1
print(f"Patch 5: Ray cache cleared ({cleared} session(s))")

print()
print("=" * 50)
print("All patches done! Now run the training cell.")
print("=" * 50)

## 10. Run PPO Training
This runs for approximately 22 hours on one L4 GPU.

In [ ]:
import os
print('Starting PPO training with Dynamic Alpha...')
print('Logs in real-time below\n')

!cd /teamspace/studios/this_studio/Emergence-of-Thinking && bash train_ppo_dynamic.sh